# Encoder Model Evaluation (LoRA Adapter)

This notebook evaluates a fine-tuned **PhayaThaiBERT** LoRA adapter on a held-out test set and logs all results to **MLflow**.

### What this notebook does
1. Loads the full label mapping from the original training data
2. Loads the quarantined test split
3. Loads the base model + LoRA adapter
4. Runs batched inference on the test set
5. Computes accuracy and collects misclassifications
6. Logs everything to MLflow — metrics, confusion matrix, per-class precision/recall/F1, and a failures artifact
7. Runs a quality gate (accuracy ≥ 50%) to validate the model

---
## 1. Install Dependencies

In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0 scikit-learn
!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"

  Cloning https://github.com/red-hat-data-services/mlflow (to revision rhoai-3.3) to /tmp/pip-req-build-4uuvb3wn
  Running command git clone --filter=blob:none --quiet https://github.com/red-hat-data-services/mlflow /tmp/pip-req-build-4uuvb3wn
  Running command git checkout -b rhoai-3.3 --track origin/rhoai-3.3
  Switched to a new branch 'rhoai-3.3'
  branch 'rhoai-3.3' set up to track 'origin/rhoai-3.3'.
  Resolved https://github.com/red-hat-data-services/mlflow to commit be6989f1f971fa67ed3bebd1e07c2cf1072bdaff
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


---
## 2. Imports

| Library | Purpose |
|---|---|
| `transformers` | Load the base encoder model and tokenizer |
| `peft` | Load the LoRA adapter on top of the base model |
| `mlflow` | Log evaluation metrics, dataset lineage, confusion matrix, failures |
| `sklearn` | `accuracy_score` for computing test accuracy |
| `pandas` | Structure predictions into a DataFrame for MLflow evaluation |
| `tqdm` | Progress bar for batched inference |

In [2]:
import json
import torch
import os
import pandas as pd
from sklearn.metrics import accuracy_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm
import mlflow
from mlflow.models import MetricThreshold

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.7.1+cu128
CUDA available: True
GPU: Tesla T4


---
## 3. MLflow Authentication

Set the MLflow tracking URI and authentication token before any MLflow operations.
- **`MLFLOW_TRACKING_TOKEN`** — your OpenShift token 
- **`MLFLOW_TRACKING_URI`** — the MLflow server endpoint

In [3]:
from getpass import getpass

os.environ["MLFLOW_TRACKING_TOKEN"] = os.environ.get("MLFLOW_TRACKING_TOKEN", "sha256~ifFt18KkwNyIzeBagf4AJ4oG-JZyai5_PO7vw2IdrgE")
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "encoder-sft"
os.environ["MLFLOW_RUN_NAME"] = "encoder-lora-eval-run"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"encoder-sft\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "encoder-model-finetuning"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


---
## 4. Configuration

| Parameter | Description |
|---|---|
| `BASE_MODEL_ID` | The Hugging Face model that was fine-tuned (must match training) |
| `ADAPTER_PATH` | Path to the saved LoRA adapter (output of training notebook) |
| `TRAINING_DATA_PATH` | Original full training data — needed to reconstruct the complete label mapping |
| `TEST_DATA_PATH` | The quarantined test split saved during training |
| `DATASET_SOURCE` | Source identifier for MLflow data lineage |
| `BATCH_SIZE` | Number of examples per inference batch |
| `RUN_NAME` | Name for the MLflow evaluation run |

In [4]:
BASE_MODEL_ID = "clicknext/phayathaibert"
ADAPTER_PATH = "/mnt/data/adapters/latest"              # <-- output from training notebook
TRAINING_DATA_PATH = "/mnt/data/training_dataset.json"
TEST_DATA_PATH = "/mnt/data/test_split.jsonl"
DATASET_SOURCE = "/mnt/data/training_dataset.json"
BATCH_SIZE = 32
RUN_NAME = "encoder-lora-eval-run"

print(f"Base model:    {BASE_MODEL_ID}")
print(f"Adapter:       {ADAPTER_PATH}")
print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Test data:     {TEST_DATA_PATH}")
print(f"Batch size:    {BATCH_SIZE}")

Base model:    clicknext/phayathaibert
Adapter:       /mnt/data/adapters/latest
Training data: /mnt/data/training_dataset.json
Test data:     /mnt/data/test_split.jsonl
Batch size:    32


---
## 5. Build Label Mapping from Full Training Data

The label-to-ID mapping **must exactly match** what was used during training. If we only look at the test split, we might miss intents that happen to not appear in the test set, which would shift the IDs and produce wrong predictions.

So we reload the full training data and extract all unique intents, sorted alphabetically (same as the training notebook).

In [5]:
print(f"Extracting full label mapping from {TRAINING_DATA_PATH}...")

with open(TRAINING_DATA_PATH, 'r') as f:
    training_data = json.load(f)

all_intents = set(row.get("intent", "UNKNOWN") for row in training_data)
unique_intents = sorted(list(all_intents))

num_labels = len(unique_intents)
id2label = {i: label for i, label in enumerate(unique_intents)}
label2id = {label: i for i, label in enumerate(unique_intents)}

print(f"Found {num_labels} unique intents:\n")
for idx, label in id2label.items():
    print(f"  {idx}: {label}")

Extracting full label mapping from /mnt/data/training_dataset.json...
Found 16 unique intents:

  0: MOBILE_BILLING_CHECK_DUE_DATE
  1: MOBILE_BILLING_PAYMENT_STATUS_POSTPAID
  2: MOBILE_BILLING_PAYMENT_STATUS_PREPAID
  3: MOBILE_NETWORK_CHECK_NO_SIGNAL
  4: MOBILE_PACKAGE_CHECK_DATA_CURRENT
  5: MOBILE_PACKAGE_CHECK_DATA_HISTORY
  6: MOBILE_ROAMING_CHECK_DATA_CURRENT
  7: MOBILE_ROAMING_CHECK_PAYMENT_POSTPAID
  8: MOBILE_SIM_ACTIVATION_POSTPAID
  9: MOBILE_SIM_ACTIVATION_PREPAID
  10: MOBILE_SIM_REPLACEMENT_POSPAID
  11: MOBILE_SIM_REPLACEMENT_PREPAID
  12: MOBILE_USAGE_CHECK_DATA_CURRENT
  13: MOBILE_USAGE_COMPARE_DATA_PLAN
  14: MOBILE_VAS_SUBSCRIBE_DATA_HISTORY
  15: MOBILE_VAS_SUBSCRIBE_DATA_SPECIFIC_MONTH


---
## 6. Load Test Data

The test set was quarantined during training (saved to `/mnt/data/test_split.jsonl`). It was **never seen** during training — this ensures our evaluation is unbiased.

If the test split file doesn't exist, we recreate it deterministically from the original data using the same split logic and seed as the training notebook.

In [6]:
if not os.path.exists(TEST_DATA_PATH):
    print(f"Warning: {TEST_DATA_PATH} not found.")
    print("Recreating the deterministic test split from the original training data...")
    from datasets import load_dataset

    dataset = load_dataset("json", data_files=TRAINING_DATA_PATH, split="train")
    split_1 = dataset.train_test_split(test_size=0.2, seed=42)
    split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

    data = [row for row in split_2["test"]]
    print(f"Recreated test split: {len(data)} examples")
else:
    print(f"Loading test data from {TEST_DATA_PATH}...")
    with open(TEST_DATA_PATH) as f:
        data = [json.loads(line) for line in f]
    print(f"Loaded {len(data)} test examples")

print(f"\nSample entry:")
data[0]

Loading test data from /mnt/data/test_split.jsonl...
Loaded 47 test examples

Sample entry:


{'user_message': 'แจ้งวันครบกำหนด pls',
 'intent': 'MOBILE_BILLING_CHECK_DUE_DATE',
 'session_history': [{'content': 'รบกวนขอทราบหมายเลขโทรศัพท์คุณลูกค้าก่อนนะคะ',
   'role': 'assistant'},
  {'content': '08xxxxxxxx', 'role': 'user'},
  {'content': 'ระบบพบยอดค้างชำระ 1 รอบค่ะ', 'role': 'assistant'},
  {'content': 'ok thx', 'role': 'user'},
  {'content': 'ยินดีึค่ะ ลูกค้าต้องการความช่วยเหลือเพิ่มเติมไหมคะ',
   'role': 'assistant'}]}

---
## 7. Load Base Model + LoRA Adapter

Two-step loading process:
1. Load the **base model** (`clicknext/phayathaibert`) with the correct number of labels and the label mapping
2. Load the **LoRA adapter** on top using `PeftModel.from_pretrained()`

The adapter file is tiny (~2-5 MB) — it only contains the trained LoRA matrices and classification head, not the full 278M parameters.

In [7]:
print(f"Loading base model {BASE_MODEL_ID} with {num_labels} labels...")

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    device_map="auto",
    trust_remote_code=True
)

print(f"Loading LoRA adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print(f"Model loaded and set to eval mode.")
print(f"Device: {model.device}")

Loading base model clicknext/phayathaibert with 16 labels...


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at clicknext/phayathaibert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading LoRA adapter from /mnt/data/adapters/latest...
Model loaded and set to eval mode.
Device: cuda:0


---
## 8. Load Tokenizer

Must use the same tokenizer as training — a mismatched tokenizer maps text to wrong token IDs.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print(f"Tokenizer loaded (vocab size: {tokenizer.vocab_size})")

Tokenizer loaded (vocab size: 248427)


---
## 9. Text Formatting Function

This function flattens the session history + current message into a single string, exactly the same way the training notebook did:

```
"User: ช่วยเช็ค roaming data | Assistant: ตอนนี้เหลือ 950MB | User: อ๋อ แล้วมันใช้ได้ถึงวันไหน"
```

**Critical:** This must match the training format exactly. If the model was trained with `"User: "` and `"Assistant: "` prefixes separated by `" | "`, evaluation must use the same format.

In [ ]:
def format_text_for_encoder(row: dict) -> str:
    """
    Flattens the conversation for BERT evaluation.
    MUST exactly match the structural formatting used during training.
    """
    history_texts = []
    
    # 1. Truncation Safety Net: Match the 3-turn limit from training
    recent_history = row.get("session_history", [])[-3:]
    
    for turn in recent_history:
        # Match the exact prefix used in training
        prefix = "Assistant: " if turn.get("role") in ["assistant", "bot"] else "User: "
        history_texts.append(prefix + turn.get("content", ""))

    history_string = " | ".join(history_texts)
    current_msg = row.get("user_message", "")
    
    # 2. The SEP Anchor: Match the structural nudges from training
    if history_string:
        return f"History: {history_string} [SEP] Current: {current_msg}"
    else:
        return f"Current: {current_msg}"

print("Sample formatted text:")
print(format_text_for_encoder(data[1]))

Sample formatted text:
Assistant: ตรวจสอบแล้วค่ะ มกราคม 2569 ไม่มีซื้อเน็ตเสริม, กุมภาพันธ์ 2569 ซื้อ 1 ครั้ง 179 บาท (10GB) และมีนาคม 2569 ซื้อ 1 ครั้ง 79 บาท (3GB) ค่ะ | User: รวมเท่าไหร่ครับ | Assistant: ค่าเน็ตเสริมรวม 3 เดือนคือ 258 บาทค่ะ หากซื้อบ่อย แนะนำอัปเกรดแพ็กเกจหลักจะประหยัดกว่าค่ะ | User: จะลองดูครับ ขอบคุณ | Assistant: ยินดีค่ะ หากต้องการข้อมูลเพิ่มเติม สอบถามได้ตลอดนะคะ | User: เดือน ม.ค. - มี.ค. เคยซื้อเน็ตเสริมมั้ยครับ กี่ บ.


---
## 10. Batched Evaluation

For each batch of test examples:
1. **Format** the text (flatten session history)
2. **Tokenize** with the same settings as training (`max_length=512`, `padding="max_length"`)
3. **Forward pass** through the model (no gradient computation needed — `torch.no_grad()`)
4. **Argmax** on logits to get predicted class index
5. Map index back to intent name using `id2label`
6. Track correct/wrong predictions and collect misclassifications

In [10]:
failures = []
y_true = []
y_pred = []

print(f"Running batched evaluation ({len(data)} examples, batch_size={BATCH_SIZE})...\n")

for i in tqdm(range(0, len(data), BATCH_SIZE)):
    batch = data[i : i + BATCH_SIZE]

    batch_texts = [format_text_for_encoder(row) for row in batch]
    batch_ground_truths = [row.get("intent", "UNKNOWN") for row in batch]

    inputs = tokenizer(
        batch_texts,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

    pred_intents = [id2label.get(pred.item(), "UNKNOWN") for pred in predictions]

    for row, ground_truth, pred_intent in zip(batch, batch_ground_truths, pred_intents):
        if pred_intent != ground_truth:
            failures.append({
                "type": "WRONG_INTENT",
                "input": row.get("user_message", ""),
                "ground_truth": ground_truth,
                "prediction": pred_intent
            })

        y_true.append(ground_truth)
        y_pred.append(pred_intent)

accuracy = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {accuracy * 100:.2f}%")
print(f"Correct:  {len(y_true) - len(failures)} / {len(y_true)}")
print(f"Failures: {len(failures)}")

Running batched evaluation (47 examples, batch_size=32)...



100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


Accuracy: 78.72%
Correct:  37 / 47
Failures: 10


### Inspect misclassifications

Review what the model got wrong — this helps identify if certain intents are confused with each other.

In [11]:
if failures:
    failures_df = pd.DataFrame(failures)
    print(f"\n{len(failures)} misclassifications:\n")
    display(failures_df)
else:
    print("No misclassifications — perfect accuracy on the test set!")


10 misclassifications:



,type,input,ground_truth,prediction
0,WRONG_INTENT,ต้องการเปิดใช้งานซิมรายเดือนครับ เบอร์ใหม่ยังใ...,MOBILE_SIM_ACTIVATION_POSTPAID,MOBILE_SIM_ACTIVATION_PREPAID
1,WRONG_INTENT,งั้นช่วยเปิดใช้งานซิมเติมเงินเบอร์นี้ให้หน่อยค...,MOBILE_SIM_ACTIVATION_PREPAID,MOBILE_SIM_ACTIVATION_POSTPAID
2,WRONG_INTENT,ขอเช็คเน็ตคงเหลือหน่อยครับ,MOBILE_USAGE_CHECK_DATA_CURRENT,MOBILE_ROAMING_CHECK_DATA_CURRENT
3,WRONG_INTENT,จริงๆ ตอนนี้เครื่องไม่มีสัญญาณครับ,MOBILE_NETWORK_CHECK_NO_SIGNAL,MOBILE_BILLING_CHECK_DUE_DATE
4,WRONG_INTENT,เน็ตในแพ็กตอนนี้เหลือเยอะไหมคะ,MOBILE_USAGE_CHECK_DATA_CURRENT,MOBILE_PACKAGE_CHECK_DATA_CURRENT
5,WRONG_INTENT,เป็นเบอร์รายเดือนครับ ผมต้องการซิมใหม่แทนซิมเด...,MOBILE_SIM_REPLACEMENT_POSPAID,MOBILE_SIM_REPLACEMENT_PREPAID
6,WRONG_INTENT,งั้นขอเทียบแพ็กให้หน่อยได้ไหมครับ ว่าถ้าเปลี่ย...,MOBILE_USAGE_COMPARE_DATA_PLAN,MOBILE_USAGE_CHECK_DATA_CURRENT
7,WRONG_INTENT,เช๊คได้เลยไหมคับ ว่าเน็ตคงเหลืออยุ่เท่าไหร่คับ,MOBILE_USAGE_CHECK_DATA_CURRENT,MOBILE_BILLING_PAYMENT_STATUS_PREPAID
8,WRONG_INTENT,check ได้เลยไหมครับ ว่า remaining data เหลือเท...,MOBILE_USAGE_CHECK_DATA_CURRENT,MOBILE_ROAMING_CHECK_DATA_CURRENT
9,WRONG_INTENT,ค่าบริการเสริม ม.ค. ถึง มี.ค. รวมกี่ บ.ครับ,MOBILE_VAS_SUBSCRIBE_DATA_HISTORY,MOBILE_PACKAGE_CHECK_DATA_HISTORY


---
## 11. Log Results to MLflow

MLflow's `evaluate()` function auto-generates:
- **Accuracy score**
- **Per-class precision, recall, F1**
- **Confusion matrix** (saved as a PNG artifact)

We also log:
- **Dataset lineage** — which data was used for evaluation
- **Failures artifact** — the full list of misclassified examples as `failures.json`

A **quality gate** validates accuracy ≥ 50% — if this fails, the model should not be promoted to production.

In [12]:
eval_df = pd.DataFrame({"prediction": y_pred, "target": y_true})

with mlflow.start_run(run_name=RUN_NAME):

    # --- Dataset lineage ---
    if DATASET_SOURCE:
        mlflow_dataset_eval = mlflow.data.from_pandas(
            pd.DataFrame(data),
            source=DATASET_SOURCE,
            name="intent_eval_data"
        )
        mlflow.log_input(mlflow_dataset_eval, context="evaluation")
        print("Dataset lineage logged.")

    # --- Auto-evaluation: accuracy, per-class precision/recall/F1, confusion matrix ---
    print("Running MLflow model evaluation...")
    result = mlflow.models.evaluate(
        data=eval_df,
        predictions="prediction",
        targets="target",
        model_type="classifier",
    )

    print(f"\nMLflow metrics:")
    for key, value in sorted(result.metrics.items()):
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

    # --- Quality gate ---
    print("\nRunning quality gate (accuracy >= 50%)...")
    try:
        mlflow.validate_evaluation_results(
            candidate_result=result,
            validation_thresholds={
                "accuracy_score": MetricThreshold(threshold=0.50, greater_is_better=True),
            }
        )
        print("Quality gate PASSED — model meets the minimum accuracy threshold.")
    except Exception as e:
        print(f"Quality gate FAILED: {e}")

    # --- Log failures ---
    if failures:
        mlflow.log_dict(failures, "failures.json")
        error_df = pd.DataFrame(failures)
        mlflow.log_table(data=error_df, artifact_file="misclassifications.json")
        print(f"\nLogged {len(failures)} misclassifications to MLflow artifacts.")
    else:
        print("\nNo failures to log.")

print("\nEvaluation complete!")

/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

Dataset lineage logged.
Running MLflow model evaluation...


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
2026/04/13 16:45:32 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as multiclass dataset, number of classes is inferred as 16. If this is incorrect, please specify the `label_list` parameter in `evaluator_config`.
2026/04/13 16:45:32 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/lat


MLflow metrics:
  accuracy_score: 0.7872
  example_count: 47
  f1_score: 0.7656
  precision_score: 0.7773
  recall_score: 0.7872

Running quality gate (accuracy >= 50%)...
Quality gate PASSED — model meets the minimum accuracy threshold.

Logged 10 misclassifications to MLflow artifacts.
🏃 View run encoder-lora-eval at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/encoder-sft/experiments/10/runs/3afb2efb2f384506ab399ea5c19f7698
🧪 View experiment at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/encoder-sft/experiments/10

Evaluation complete!


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

<Figure size 1050x700 with 0 Axes>